# Cat Classification Experiment: Coquita vs. Ori

This notebook explores a custom CNN architecture to classify images of two cats: **Coquita** and **Ori**.
We use MLflow for experiment tracking and ensure balanced datasets for training and evaluation.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
import mlflow
import mlflow.pytorch
from mlflow.models import infer_signature
import os
import random
import numpy as np
from tqdm import tqdm
from PIL import Image

# Set seeds for reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

## 1. Data Preparation

We'll load images using `ImageFolder`, balance the classes based on the minority class (Ori), and split into Train, Validation, and Test sets.

In [2]:
DATA_DIR = '../data/raw'
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform, allow_empty=True)
print(f"Found classes: {full_dataset.classes}")
print(f"Class to index: {full_dataset.class_to_idx}")

# Balance and Remap labels to 0 and 1
coquita_class_idx = full_dataset.class_to_idx['coquita']
ori_class_idx = full_dataset.class_to_idx['ori']

coquita_idx = [i for i, (_, label) in enumerate(full_dataset.samples) if label == coquita_class_idx]
ori_idx = [i for i, (_, label) in enumerate(full_dataset.samples) if label == ori_class_idx]

min_samples = min(len(coquita_idx), len(ori_idx))
print(f"Balancing to {min_samples} samples per class.")

balanced_idx = random.sample(coquita_idx, min_samples) + random.sample(ori_idx, min_samples)

# Custom Subset to remap labels on the fly
class BinaryCatSubset(Subset):
    def __getitem__(self, idx):
        img, label = super().__getitem__(idx)
        # Remap: coquita -> 0, ori -> 1
        new_label = 0 if label == coquita_class_idx else 1
        return img, new_label

    def __getitems__(self, indices):
        return [self[i] for i in indices]

balanced_dataset = BinaryCatSubset(full_dataset, balanced_idx)

# Split: 70% Train, 15% Val, 15% Test
train_size = int(0.7 * len(balanced_dataset))
val_size = int(0.15 * len(balanced_dataset))
test_size = len(balanced_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(balanced_dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")

Found classes: ['coquita', 'false_positives', 'ori', 'other_cat', 'unlabeled']
Class to index: {'coquita': 0, 'false_positives': 1, 'ori': 2, 'other_cat': 3, 'unlabeled': 4}
Balancing to 769 samples per class.
Train size: 1076, Val size: 230, Test size: 232


## 2. Custom Neural Network: CatNet

A simple CNN architecture for binary classification.

In [3]:
class CatNet(nn.Module):
    def __init__(self):
        super(CatNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 128),  # 224 / 2 / 2 / 2 = 28
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = CatNet().to(device)
print(model)

CatNet(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=100352, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


## 3. Training and MLflow Tracking

In [4]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 10

import sys
sys.path.append('..')
from src.common.config import MLFLOW_TRACKING_URI

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("Cat_Classification_Custom")

with mlflow.start_run():
    mlflow.log_param("architecture", "CatNet")
    mlflow.log_param("lr", 0.001)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
        
        epoch_loss = running_loss / len(train_dataset)
        mlflow.log_metric("train_loss", epoch_loss, step=epoch)
        
        # Validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
                outputs = model(images)
                preds = (outputs > 0.5).float()
                val_correct += (preds == labels).sum().item()
        
        val_acc = val_correct / len(val_dataset)
        mlflow.log_metric("val_acc", val_acc, step=epoch)
        print(f"Epoch {epoch+1}: Loss {epoch_loss:.4f}, Val Acc {val_acc:.4f}")

    # Final Evaluation
    test_correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
            outputs = model(images)
            preds = (outputs > 0.5).float()
            test_correct += (preds == labels).sum().item()
    
    test_acc = test_correct / len(test_dataset)
    mlflow.log_metric("test_acc", test_acc)
    print(f"Final Test Accuracy: {test_acc:.4f}")
    
    # Log model
    mlflow.pytorch.log_model(model, name="cat_net_model")

2026/05/12 11:06:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/12 11:06:33 INFO mlflow.store.db.utils: Updating database tables
2026/05/12 11:06:33 INFO mlflow.tracking.fluent: Experiment with name 'Cat_Classification_Custom' does not exist. Creating a new experiment.
Epoch 1/10: 100%|██████████| 34/34 [00:04<00:00,  6.90it/s]


Epoch 1: Loss 0.4741, Val Acc 0.9391


Epoch 2/10: 100%|██████████| 34/34 [00:04<00:00,  7.54it/s]


Epoch 2: Loss 0.2175, Val Acc 0.9609


Epoch 3/10: 100%|██████████| 34/34 [00:04<00:00,  7.54it/s]


...

Epoch 10/10: 100%|██████████| 34/34 [00:04<00:00,  7.63it/s]


Epoch 10: Loss 0.0314, Val Acc 0.9870


2026/05/12 11:07:23 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Final Test Accuracy: 0.9569


2026/05/12 11:07:26 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
